# EV Adoption, Clean Energy, and Transport Emissions
### DATA 5310 Final Project

Team: Devlin, Naomi, Prithika, Njenga

Our question: Do countries with higher EV adoption have lower transport emissions, and does it matter how clean their electricity is?

In this notebook we combine the four datasets each of us cleaned separately into one dataset that we'll use for our visualizations. All of the data comes from Our World in Data. We focused on 10 countries: Norway, China, the United States, Germany, the United Kingdom, Brazil, Thailand, Japan, India, and South Africa. We picked these to get a mix of regions and a range of EV adoption levels.

| Dataset | Column(s) | Unit | Years |
|---|---|---|---|
| Share of new cars sold that are electric | ev_share | % of new car sales | 2010 to 2025 |
| CO2 emissions from transport per capita | transport_co2 | kg per person | 2010 to 2023 |
| Fossil fuel consumption per capita | gas, oil, coal, total, and fuel shares | kWh per person and % | 2010 to 2025 |
| Share of electricity from renewables | renewable_share | % of electricity | 2010 to 2025 |


### Step 1: Load the data

Each of us saved our cleaned dataset as a CSV. Here we load all four.

In [1]:
import pandas as pd

In [2]:
# load the required datasets
emission = pd.read_csv("Emission_transport_cleaned.csv")
renewable = pd.read_csv("renewables_cleaned_1.csv")
fossil_fuel = pd.read_csv("fossil_fuel_data_cleaned.csv")
ev_sales = pd.read_csv("ev_share_clean.csv")


In [3]:
fossil_fuel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   entity      160 non-null    object 
 1   code        160 non-null    object 
 2   year        160 non-null    int64  
 3   gas_kwh     160 non-null    float64
 4   oil_kwh     160 non-null    float64
 5   coal_kwh    160 non-null    float64
 6   total_kwh   160 non-null    float64
 7   gas_share   160 non-null    float64
 8   oil_share   160 non-null    float64
 9   coal_share  160 non-null    float64
dtypes: float64(7), int64(1), object(2)
memory usage: 12.6+ KB


In [4]:
renewable.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 465 entries, 0 to 464
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  465 non-null    int64  
 1   Entity      465 non-null    object 
 2   Code        465 non-null    object 
 3   Year        465 non-null    int64  
 4   Renewables  465 non-null    float64
dtypes: float64(1), int64(2), object(2)
memory usage: 18.3+ KB


In [5]:
emission.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Entity                    140 non-null    object 
 1   Code                      140 non-null    object 
 2   Year                      140 non-null    int64  
 3   Emissions from transport  140 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 4.5+ KB


In [6]:
ev_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Entity    157 non-null    object 
 1   Code      157 non-null    object 
 2   Year      157 non-null    int64  
 3   ev_share  157 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 5.0+ KB


In [7]:
renewable["Entity"].unique()

array(['Brazil', 'China', 'Germany', 'India', 'Japan', 'Norway',
       'South Africa', 'Thailand', 'United Kingdom', 'United States'],
      dtype=object)

### Step 2: Fix column names

A few things had to be fixed before we could merge. The fossil fuel data used lowercase column names (entity, code, year), so we changed them to match the other datasets. The renewables file had an extra unnamed column from being saved with its index, so we dropped it. We also shortened some of the long column names to make them easier to work with.

In [8]:
# Make fossil fuel columns match the others
fossil_fuel = fossil_fuel.rename(columns={"entity": "Entity", "code": "Code", "year": "Year"})

# Drop the extra index column from renewables
renewable = renewable.drop(columns=["Unnamed: 0"])

# Rename value columns so they're short and clear
renewable = renewable.rename(columns={"Renewables": "renewable_share"})
emission = emission.rename(columns={"Emissions from transport": "transport_co2"})

### Step 3: Merge

We merged the four datasets on Entity, Code, and Year using an outer merge. We went with outer because the datasets don't all cover the same years, and we didn't want to lose any rows. After merging, we kept only 2010 to 2025 and sorted by country and year.

In [9]:
# Merge all four datasets, keeping every row
merged = ev_sales.merge(emission, on=["Entity", "Code", "Year"], how="outer")
merged = merged.merge(fossil_fuel, on=["Entity", "Code", "Year"], how="outer")
merged = merged.merge(renewable, on=["Entity", "Code", "Year"], how="outer")

# Keep only 2010 to 2025
merged = merged[(merged["Year"] >= 2010) & (merged["Year"] <= 2025)]
merged = merged.sort_values(["Entity", "Year"]).reset_index(drop=True)

merged.shape

(160, 13)

### Step 4: Check the merged data

The merged data has 160 rows and 13 columns, which is 10 countries over 16 years. There are some missing values we expected. The transport CO2 data stops at 2023, so 2024 and 2025 are empty for that column. South Africa's EV data starts in 2013, so 2010 to 2012 are empty for ev_share. We kept these rows so the other variables still show the latest years. For charts that use transport CO2, we'll stop at 2023.

In [10]:
# Check the results
print(merged["Entity"].nunique())
merged.isna().sum()

10


Entity              0
Code                0
Year                0
ev_share            3
transport_co2      20
gas_kwh             0
oil_kwh             0
coal_kwh            0
total_kwh           0
gas_share           0
oil_share           0
coal_share          0
renewable_share     0
dtype: int64

In [11]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Entity           160 non-null    object 
 1   Code             160 non-null    object 
 2   Year             160 non-null    int64  
 3   ev_share         157 non-null    float64
 4   transport_co2    140 non-null    float64
 5   gas_kwh          160 non-null    float64
 6   oil_kwh          160 non-null    float64
 7   coal_kwh         160 non-null    float64
 8   total_kwh        160 non-null    float64
 9   gas_share        160 non-null    float64
 10  oil_share        160 non-null    float64
 11  coal_share       160 non-null    float64
 12  renewable_share  160 non-null    float64
dtypes: float64(10), int64(1), object(2)
memory usage: 16.4+ KB
